pip install llama_index==0.10.20   
pip install llama-index-embeddings-huggingface==0.1.4   
pip install llama-index-llms-huggingface==0.1.4   
pip install llama-index-core==0.10.20.post2  
   
cd BCEmbedding  
pip install -v -e .  

In [1]:
from BCEmbedding.tools.llama_index import BCERerank

In [2]:
import os

In [3]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [4]:
from llama_index.core import VectorStoreIndex, ServiceContext, SimpleDirectoryReader

In [5]:
from llama_index.core.node_parser import SimpleNodeParser

In [6]:
from llama_index.core.retrievers import VectorIndexRetriever

In [7]:
# 初始化 embedding模型 和 reranker模型

embed_args = {"model_name": '/root/project/models/bce-embedding-base_v1', 
              "max_length": 512,
              "embed_batch_size": 32}

embed_model = HuggingFaceEmbedding(**embed_args)

In [8]:
reranker_args = {"model": "/root/project/models/bce-reranker-base_v1",
                  "top_n": 10,}

reranker_model = BCERerank(**reranker_args)

03/18/2024 03:18:09 - [INFO] -BCEmbedding.models.RerankerModel->>>    Loading from `/root/project/models/bce-reranker-base_v1`.
03/18/2024 03:18:09 - [INFO] -BCEmbedding.models.RerankerModel->>>    Execute device: cuda;	 gpu num: 1;	 use fp16: False


In [9]:
# 提取embeddings

query = "苹果"

passages = [
    "我喜欢苹果",
    "我喜欢橘子",
    "苹果和橘子都是水果"
]

query_embedding = embed_model.get_query_embedding(query)
print("query_embedding shape: ", len(query_embedding))

passages_embeddings = embed_model.get_text_embedding_batch(passages)
print("passages_embeddings shape: ", len(passages_embeddings))


query_embedding shape:  768
passages_embeddings shape:  3


In [10]:
# llm

from llama_index.llms.huggingface import HuggingFaceLLM
import torch

from llama_index.core import PromptTemplate

qa_template = PromptTemplate(
    "以下是描述一个任务的指令。"
    "请根据用户输入的内容，一步一步地回答。\n\n"
    "### 指令:\n{query_str}\n\n### 回复:"
)

llm = HuggingFaceLLM(
    context_window=12000,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.25, "do_sample": False},
    query_wrapper_prompt=qa_template,
    tokenizer_name="/root/project/models/Qwen/Qwen1.5-7B-Chat",
    model_name="/root/project/models/Qwen/Qwen1.5-7B-Chat",
    device_map="auto",
    tokenizer_kwargs={"max_length": 512},
    # uncomment this if using CUDA to reduce memory usage
    model_kwargs={"torch_dtype": torch.float16}
)

03/18/2024 03:18:10 - [INFO] -accelerate.utils.modeling->>>    We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [11]:
service_context = ServiceContext.from_defaults(llm=llm, embed_model=embed_model)

/tmp/ipykernel_4064/3999105131.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  service_context = ServiceContext.from_defaults(llm=llm, embed_model=embed_model)


In [12]:
documents = SimpleDirectoryReader(input_files=["files/test.pdf"]).load_data()

In [13]:
len(documents)

52

In [14]:
documents[0]

Document(id_='8185dd72-d073-4d92-a128-cf1412be8e56', embedding=None, metadata={'page_label': '1', 'file_name': 'test.pdf', 'file_path': 'files/test.pdf', 'file_type': 'application/pdf', 'file_size': 5491765, 'creation_date': '2024-03-18', 'last_modified_date': '2024-03-18'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text='互联网创作者经济白皮书©2022.8 iResearch Inc. \n', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n')

In [15]:
node_parser = SimpleNodeParser.from_defaults(chunk_size=1500, chunk_overlap=200)

In [16]:
nodes = node_parser.get_nodes_from_documents(documents)

In [17]:
nodes[0]

TextNode(id_='73a875e6-5685-4b53-90df-26873a1126db', embedding=None, metadata={'page_label': '1', 'file_name': 'test.pdf', 'file_path': 'files/test.pdf', 'file_type': 'application/pdf', 'file_size': 5491765, 'creation_date': '2024-03-18', 'last_modified_date': '2024-03-18'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='8185dd72-d073-4d92-a128-cf1412be8e56', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '1', 'file_name': 'test.pdf', 'file_path': 'files/test.pdf', 'file_type': 'application/pdf', 'file_size': 5491765, 'creation_date': '2024-03-18', 'last_modified_date': '2024-03-18'}, hash='2ceecb907e42164d858f5db919a490790738f01d2e58f6d5befdf03d3160387d'), <NodeRelationship.NEXT: '3'>: Rela

In [18]:
index = VectorStoreIndex(nodes, service_context=service_context)

In [19]:
query = "万兴科技做什么的?"

In [20]:
vector_retriever = VectorIndexRetriever(index=index, similarity_top_k=3, service_context=service_context)

In [21]:
retrieval_by_embedding = vector_retriever.retrieve(query)

In [22]:
for item in retrieval_by_embedding:
    print(item.text + "\n\n")
    print("-"*100)

33 ©2022.8 iResearch Inc.                                                                                                       www.iresearch.com.cn
互联网内容创作工具及服务典型企业案例万兴科技：中国数字创意软件龙头，以创作者为中心，创意工具与资源并驾齐驱•万兴科技是全球领先的新生代数字创意赋能者，成立于2003年，面向全球海量新生代互联网用户，提供简单高效的数字创意软件、潮流时尚的创意资源和丰富多元的生态化服务，旗下产品覆盖视频创意（万兴喵影、WondershareFilmora、WondershareVidAir、StoryChic、VidChic、万兴优转、万兴录演等）、绘图创意（亿图图示、亿图脑图MindMaster、墨刀等）、文档创意（万兴PDF、WondersharePDFelement、WondershareDocument Cloud等）、实用工具（万兴恢复专家、万兴数据管家、万兴易修、WondershareDr.Fone等）等多个领域，业务范围遍及全球200 多个国家和地区，用户活跃数超1亿，累计用户数超15亿。
来源：万兴科技，及其他公开资料，艾瑞咨询研究院自主研究及绘制。布局全栈式视频创意工具，围绕创作者提供生产工具、创意资源及培训赋能等
创意资源平台建设结合AI等智能化技术，实现素材智能识别、标记等功能，深化全球全链条正版素材库布局，打造坚实的内容底层生态平台，建立“数字原料工厂”
创作者生产工具创意资源创作者能力扶持创作基地建设
创作者平台工具及服务
Sandbox万兴科技元宇宙创作者俱乐部视频创意
文档创意
万兴喵影
SweetSelfieBeat.ly……
万兴优转
亿图图示亿图脑图MindMaster绘图创意
Pixso……
万兴PDF
WondersharePDFConverter……桌面端+Web+
移动端
StoryChic
墨刀


----------------------------------------------------------------------------------------------------
20 ©2022.8 iResearch Inc.

In [23]:
retrieval_by_reranker = reranker_model.postprocess_nodes(retrieval_by_embedding, query_str=query)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [24]:
for item in retrieval_by_reranker:
    print(item.text + "\n\n")
    print("-"*100)

33 ©2022.8 iResearch Inc.                                                                                                       www.iresearch.com.cn
互联网内容创作工具及服务典型企业案例万兴科技：中国数字创意软件龙头，以创作者为中心，创意工具与资源并驾齐驱•万兴科技是全球领先的新生代数字创意赋能者，成立于2003年，面向全球海量新生代互联网用户，提供简单高效的数字创意软件、潮流时尚的创意资源和丰富多元的生态化服务，旗下产品覆盖视频创意（万兴喵影、WondershareFilmora、WondershareVidAir、StoryChic、VidChic、万兴优转、万兴录演等）、绘图创意（亿图图示、亿图脑图MindMaster、墨刀等）、文档创意（万兴PDF、WondersharePDFelement、WondershareDocument Cloud等）、实用工具（万兴恢复专家、万兴数据管家、万兴易修、WondershareDr.Fone等）等多个领域，业务范围遍及全球200 多个国家和地区，用户活跃数超1亿，累计用户数超15亿。
来源：万兴科技，及其他公开资料，艾瑞咨询研究院自主研究及绘制。布局全栈式视频创意工具，围绕创作者提供生产工具、创意资源及培训赋能等
创意资源平台建设结合AI等智能化技术，实现素材智能识别、标记等功能，深化全球全链条正版素材库布局，打造坚实的内容底层生态平台，建立“数字原料工厂”
创作者生产工具创意资源创作者能力扶持创作基地建设
创作者平台工具及服务
Sandbox万兴科技元宇宙创作者俱乐部视频创意
文档创意
万兴喵影
SweetSelfieBeat.ly……
万兴优转
亿图图示亿图脑图MindMaster绘图创意
Pixso……
万兴PDF
WondersharePDFConverter……桌面端+Web+
移动端
StoryChic
墨刀


----------------------------------------------------------------------------------------------------
20 ©2022.8 iResearch Inc.

### 查询引擎：方式-1

In [25]:
query_engine = index.as_query_engine(node_postprocessors=[reranker_model])

In [26]:
query_response = query_engine.query(query)

/home/vipuser/miniconda3/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.25` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/vipuser/miniconda3/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/vipuser/miniconda3/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:427: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


In [27]:
print(str(query_response))

 万兴科技是一家专注于数字创意软件和服务的公司。它提供各种创新的工具，包括视频创意软件、绘图创意工具、文档创意工具以及实用工具，满足全球互联网用户的需求。万兴科技的产品线覆盖多个领域，如视频编辑（如喵影、Filmora等）、图像处理（亿图图示、MindMaster等）、PDF解决方案（如PDFelement）以及数据管理工具。此外，他们还致力于支持创作者，通过提供创意资源、培训赋能和创作平台来促进内容创作。


### 查询引擎：方式-2

In [28]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(vector_retriever, llm=llm)

In [29]:
response = query_engine.query(query)

In [30]:
print(str(response))

 万兴科技是一家专注于数字创意软件的公司，提供简单高效的数字创意工具，包括视频创意、绘图创意、文档创意以及实用工具等。它以创作者为中心，提供创意工具和资源，并且布局了全栈式视频创意工具，支持创作者在PC和Web端进行内容创作。此外，万兴科技还通过AI技术来建设素材智能识别和标记功能，拥有全球全链条正版素材库，致力于打造内容底层生态平台。他们的业务覆盖全球多个国家和地区，有大量的用户活跃和付费情况的数据。
